# Optimize spatial synthetic data parameters

This notebook searches for a spatial synthetic data-generating process such that:

- the **IID** engine declares equivalence,
- the **cluster-aware** engine declares equivalence, and
- the **spatial** engine does **not** declare equivalence.

The notebook follows the shared optimization flow used across the synthetic-data notebooks.


## 1. Environment setup


In [ ]:
from pyTOST.data_gen.notebook_utils import (
    configure_notebook_environment,
    summarize_result,
    history_frame,
    result_payload,
)
from pyTOST.data_gen.optimize_spatial_params import SearchSpace, search, save_best

REPO_ROOT = configure_notebook_environment()
ALPHA = 0.05
MARGIN = 1.0
SEARCH_SEED = 123
VALIDATION_SEED = 2026
OUTPUT_JSON = "best_spatial_params.json"

ENGINE_SPECS = {
    "IID": {"ci": "ci_iid", "eq": "eq_iid", "mu": "mu_hat_iid"},
    "Cluster": {"ci": "ci_cluster", "eq": "eq_cluster", "mu": "mu_hat_cluster"},
    "Spatial": {"ci": "ci_spatial", "eq": "eq_spatial", "mu": "mu_hat_spatial"},
}


## 2. Configure the optimization problem


In [ ]:
space = SearchSpace()
N_ITER = 400
VERBOSE_EVERY = 25


## 3. Run the search


In [ ]:
best, history = search(
    seed=SEARCH_SEED,
    n_iter=N_ITER,
    alpha=ALPHA,
    margin=MARGIN,
    space=space,
    verbose_every=VERBOSE_EVERY,
)

best_summary = summarize_result(best, ENGINE_SPECS)
best_summary


## 4. Summarize the best candidate


In [ ]:
best_payload = result_payload(
    params=best.params,
    summary_source=best,
    engine_specs=ENGINE_SPECS,
)
best_payload


## 5. Save the best parameters


In [ ]:
save_best(best, OUTPUT_JSON, alpha=ALPHA, margin=MARGIN)
OUTPUT_JSON


## 6. Validate by regenerating data and rerunning the target engines


In [ ]:
from pyTOST.data_gen.optimize_spatial_params import evaluate_params

validation = evaluate_params(best.params, seed=VALIDATION_SEED, alpha=ALPHA, margin=MARGIN)
validation_summary = summarize_result(validation, ENGINE_SPECS)
validation_summary


## 7. Search diagnostics


In [ ]:
history_df = history_frame(
    history,
    ENGINE_SPECS,
    param_keys=["n_clusters", "points_per_cluster", "length_scale", "delta", "seed"],
)
history_df.sort_values("score").head(10)
